In [14]:
import numpy as np
import pandas as pd
from scipy.integrate import odeint

In [15]:
#Sabotage script
#bioprocess model is kept as previously
def bioprocess_model(y, t, params):
    #dependent variables definition
    X, G, L, A, P = y
    #parameters definition (max growth rate, reaction rate constant, conversion rate)
    mu_max, Ks, Ki, Y_xg, Y_lg, Y_ag, q_p = params

    #equations for growth, consumption and production
    mu = mu_max * (G / (Ks + G)) * (Ki / (Ki + A))
    
    dXdt = mu * X
    dGdt = -(1/Y_xg) * mu * X
    dLdt = Y_lg * (1/Y_xg) * mu * X
    dAdt = Y_ag * (1/Y_xg) * mu * X
    dPdt = q_p * X if t > 72 else 0
    
    return [dXdt, dGdt, dLdt, dAdt, dPdt]

def generate_sabotaged_data(num_batches=50):
    all_telemetry = []
    all_outcomes = []
    t_eval = np.linspace(0, 240, 241) 

    #defining batches
    for i in range(num_batches):
        batch_id = f"BATCH_{i:03d}"
        status = "Golden"

        #default values for parameters and initial value for parameters
        params = [0.04, 0.5, 15.0, 0.4, 0.6, 0.1, 0.05]
        y0 = [0.5, 50.0, 0.0, 0.0, 0.0]

        #ouptu value between 0.0 and 1.0
        dice_roll = np.random.random()

        # as the dice roll should have equal probability for all results,
        # there is a 15 % chance to start with lower glucose
        if dice_roll < 0.15: 
            status = "OOS_Glucose_Fail"
            y0[1] = 15.0 

        # 15 % chance to have a toxic drift, 
        # Yag of 0.4 instead of 0.1 means that for the same amount of biomass
        # there is 4 times more ammonia produced
        elif dice_roll < 0.30: 
            status = "OOS_Toxic_Drift"
            params[5] = 0.4 
            
        # 10 % chance of transfection fail,
        # low transfection is translated by low productivity rate (10 times lower)
        elif dice_roll < 0.40: 
            status = "OOS_Transfection_Fail"
            params[6] = 0.005 
            
        #solving equation system    
        sol = odeint(bioprocess_model, y0, t_eval, args=(params,))

        #creating random normaly distributed noise for all data
        noise = np.random.normal(0, 0.015, sol.shape)# random noise centered at 0 with 1.5 % standard deviation populated accross the shape of the sol matrix
        sol_noisy = sol + (sol * noise) #adding the noise fitted to the actual data with the data
        
        batch_df = pd.DataFrame(sol_noisy, columns=['VCD', 'Glucose', 'Lactate', 'Ammonia', 'Product'])
        batch_df['Hour'] = t_eval
        batch_df['Batch_ID'] = batch_id
        all_telemetry.append(batch_df)

        base_eff = 0.35
        if status == "OOS_Toxic_Drift":
            eff = base_eff * 0.6
        elif status == "OOS_Transfection_Fail":
            eff = base_eff * 0.3
        else:
            eff = base_eff * np.random.normal(1, 0.05)

        total_titer = sol[-1, 4] #fetching the last value for the 5th variable
        full_titer = total_titer * eff
        pct_full = eff * 100

        all_outcomes.append({
            'Batch_ID': batch_id,
            'Total_Titer': total_titer,
            'Full_Titer': full_titer,
            'Percent_Full': pct_full,
            'Status': status #adding the label to the batch
        })
        
    return pd.concat(all_telemetry), pd.DataFrame(all_outcomes)

def generate_dirty_data(telemetry_df):
    dirty_rows = []
    
    for batch_id in telemetry_df['Batch_ID'].unique():
        batch_data = telemetry_df[telemetry_df['Batch_ID'] == batch_id]
        
        # 1. High Frequency: Ammonia & Lactate (Every 2 hours)
        # We simulate probe data with jittery timestamps
        ammonia_data = batch_data.iloc[::2][['Hour', 'Ammonia', 'Lactate', 'Batch_ID']].copy()
        ammonia_data['Hour'] += np.random.normal(0, 0.1, len(ammonia_data)) 
        
        # 2. Low Frequency: VCD & Glucose (Every 24 hours)
        # Mimics a technician coming in once a day
        vcd_data = batch_data.iloc[::24][['Hour', 'VCD', 'Glucose', 'Batch_ID']].copy()
        vcd_data['Hour'] += np.random.normal(0, 0.5, len(vcd_data)) 
        
        # Combine into a "messy" long-form table
        dirty_rows.append(ammonia_data)
        dirty_rows.append(vcd_data)
        
    return pd.concat(dirty_rows).sort_values(['Batch_ID', 'Hour'])

df_telemetry, df_outcomes = generate_sabotaged_data(50)
df_messy = generate_dirty_data(telemetry_batches)

In [10]:
df_messy.to_csv('telemetry_batches.csv', index=False)
df_outcomes.to_csv('batch_outcomes.csv', index=False)